# PI3Kα (4JPS) — Step 6: Reference redocking (Colab Python 3.13 compatible)

Uses the official AutoDock Vina **1.2.7 Linux executable**, rather than the Python `vina` package (which lacks a Python 3.13 prebuilt wheel). The experimental ligand coordinates are used only for RMSD validation. Upload the same three ZIP inputs.


In [1]:
# Use the official Vina 1.2.7 Linux binary: Colab Python 3.13 has no vina 1.2.7 wheel.
%pip -q install rdkit meeko gemmi scipy numpy pandas
from pathlib import Path
import sys, json, zipfile, shutil, subprocess, hashlib, re, os
import requests
import numpy as np
import pandas as pd
from rdkit import Chem
from google.colab import files
from importlib.metadata import version

VINA_EXEC = Path('/content/vina_1.2.7_linux_x86_64')
VINA_URL = 'https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.7/vina_1.2.7_linux_x86_64'
if not VINA_EXEC.exists():
    response = requests.get(VINA_URL, timeout=120, allow_redirects=True)
    response.raise_for_status()
    VINA_EXEC.write_bytes(response.content)
VINA_EXEC.chmod(0o755)
check = subprocess.run([str(VINA_EXEC), '--version'], capture_output=True, text=True)
print('Python:', sys.version.split()[0])
print('Vina executable version:', (check.stdout + check.stderr).strip())
print('RDKit:', version('rdkit'), 'Meeko:', version('meeko'))
if check.returncode != 0 or '1.2.7' not in (check.stdout + check.stderr):
    raise RuntimeError('Could not validate official AutoDock Vina 1.2.7 executable')
VINA_VERSION = '1.2.7'


Python: 3.13.18
Vina executable version: AutoDock Vina v1.2.7
RDKit: 2026.9.2 Meeko: 0.8.0


## 1. Upload exactly three input ZIP archives

Choose all three at once in the Colab upload dialog. Inputs are extracted by *expected file content*, not by ZIP filename, so Colab-renamed `(1)` files are accepted.

In [3]:

from pathlib import Path
from google.colab import files
from rdkit import Chem
import zipfile

out = Path("PI3Kalpha_4JPS_Step6")
out.mkdir(exist_ok=True)

print("Upload the prepared receptor, Step 4, and Step 5 ZIP files:")
uploaded = files.upload()

zipnames = [
    name for name in uploaded
    if name.lower().endswith(".zip")
]

if len(zipnames) != 3:
    raise RuntimeError(
        f"Upload exactly 3 ZIPs; got {zipnames}"
    )

needed = {
    "4JPS_AB_distance_filtered.pdbqt": "receptor",
    "1LT_reference.pdbqt": "ligand",
    "1LT_crystal_heavy_atom_mapped.sdf": "crystal",
    "receptor_distance_filtered.log": "preparation_log",
    "pi3ka_step4_ligand_mapping_report.json": "mapping_report"
}

found = {}

for zname in zipnames:
    with zipfile.ZipFile(zname) as z:

        bad_file = z.testzip()
        if bad_file is not None:
            raise RuntimeError(
                f"Corrupted file {bad_file} in {zname}"
            )

        for member in z.namelist():
            base = Path(member).name

            if base not in needed:
                continue

            incoming = z.read(member)

            # Handle identical files included in multiple ZIPs.
            if base in found:
                existing = (out / base).read_bytes()

                if existing != incoming:
                    raise ValueError(
                        f"Conflicting versions of {base} "
                        f"in {found[base]} and {zname}"
                    )

                print(f"Identical duplicate skipped: {base}")
                continue

            # Save the first verified copy.
            (out / base).write_bytes(incoming)
            found[base] = zname

missing = set(needed) - set(found)

if missing:
    raise FileNotFoundError(
        f"Missing inputs: {sorted(missing)}"
    )

print("\nSuccessfully extracted input files:")

for filename, source in found.items():
    print(
        filename,
        "<-", source,
        "bytes:", (out / filename).stat().st_size
    )

# Validate crystallographic reference ligand.
reference = next(
    (
        mol for mol in Chem.SDMolSupplier(
            str(out / "1LT_crystal_heavy_atom_mapped.sdf"),
            removeHs=True
        )
        if mol is not None
    ),
    None
)

assert reference is not None, "Crystal SDF could not be read"
assert reference.GetNumHeavyAtoms() == 30, \
    "Reference ligand heavy-atom count mismatch"

# Validate ligand PDBQT.
ligand_text = (out / "1LT_reference.pdbqt").read_text()

assert "REMARK SMILES" in ligand_text, \
    "Meeko mapping remarks missing from ligand PDBQT"

# Validate receptor PDBQT.
receptor_text = (
    out / "4JPS_AB_distance_filtered.pdbqt"
).read_text()

receptor_atom_count = sum(
    line.startswith(("ATOM  ", "HETATM"))
    for line in receptor_text.splitlines()
)

assert receptor_atom_count > 100, \
    "Receptor PDBQT appears incomplete"

print("\nPASS: All redocking inputs passed basic format checks")
print("Reference ligand heavy atoms:", reference.GetNumHeavyAtoms())
print("Receptor atom records:", receptor_atom_count)
print("Ready for the next Step 6 cell.")


Upload the prepared receptor, Step 4, and Step 5 ZIP files:


Saving PI3Kalpha_4JPS_Prepared_Receptor.zip to PI3Kalpha_4JPS_Prepared_Receptor (1).zip
Saving PI3Kalpha_4JPS_Step4.zip to PI3Kalpha_4JPS_Step4 (1).zip
Saving PI3Kalpha_4JPS_Step5.zip to PI3Kalpha_4JPS_Step5 (1).zip
Identical duplicate skipped: 1LT_crystal_heavy_atom_mapped.sdf
Identical duplicate skipped: pi3ka_step4_ligand_mapping_report.json

Successfully extracted input files:
4JPS_AB_distance_filtered.pdbqt <- PI3Kalpha_4JPS_Prepared_Receptor (1).zip bytes: 986880
receptor_distance_filtered.log <- PI3Kalpha_4JPS_Prepared_Receptor (1).zip bytes: 5874
pi3ka_step4_ligand_mapping_report.json <- PI3Kalpha_4JPS_Step4 (1).zip bytes: 677
1LT_crystal_heavy_atom_mapped.sdf <- PI3Kalpha_4JPS_Step4 (1).zip bytes: 2625
1LT_reference.pdbqt <- PI3Kalpha_4JPS_Step5 (1).zip bytes: 3184

PASS: All redocking inputs passed basic format checks
Reference ligand heavy atoms: 30
Receptor atom records: 12336
Ready for the next Step 6 cell.


## 2. Provisional docking box and Vina run

Grid center and size are sourced from the crystallographic 1LT extent with 6 Å margin on each side. **Do not treat this as validated until RMSD assessment.** Docking may take several minutes on a free Colab CPU; keep the cell running until it finishes.

In [4]:
# Execute AutoDock Vina 1.2.7 directly (no Python vina binding needed).
BOX_CENTER=[-1.319,-9.513,16.948]
BOX_SIZE=[18.946,19.659,25.114]
SEED=42; EXHAUSTIVENESS=32; N_POSES=9
poses_path=out/'1LT_redocked_all_poses.pdbqt'
log_path=out/'vina_redocking.log'
cmd = [str(VINA_EXEC),
       '--receptor',str(out/'4JPS_AB_distance_filtered.pdbqt'),
       '--ligand',str(out/'1LT_reference.pdbqt'),
       '--center_x',str(BOX_CENTER[0]),'--center_y',str(BOX_CENTER[1]),'--center_z',str(BOX_CENTER[2]),
       '--size_x',str(BOX_SIZE[0]),'--size_y',str(BOX_SIZE[1]),'--size_z',str(BOX_SIZE[2]),
       '--exhaustiveness',str(EXHAUSTIVENESS),'--num_modes',str(N_POSES),
       '--seed',str(SEED),'--cpu','2','--scoring','vina',
       '--out',str(poses_path)]
print('Running official Vina executable...')
p=subprocess.run(cmd, capture_output=True, text=True)
log=(p.stdout or '')+'\n'+(p.stderr or '')
log_path.write_text('COMMAND: '+' '.join(cmd)+'\n\n'+log,encoding='utf-8')
print(log[-9000:])
if p.returncode!=0 or not poses_path.exists() or poses_path.stat().st_size==0:
    raise RuntimeError(f'Redocking failed, return code {p.returncode}; inspect vina_redocking.log')
# Vina writes per-pose energies to its PDBQT REMARK lines.
energies=[]
for line in poses_path.read_text().splitlines():
    if line.startswith('REMARK VINA RESULT:'):
        values=line.split(':',1)[1].split()
        energies.append((float(values[0]),))
if not energies:
    raise RuntimeError('No VINA RESULT energies found in docking output')
print('Saved',poses_path,'poses:',len(energies))
df_energies=pd.DataFrame([{'pose':i+1,'vina_total_kcal_mol':float(e[0])} for i,e in enumerate(energies)])
df_energies.to_csv(out/'redocking_affinities.csv',index=False)
display(df_energies)


Running official Vina executable...
AutoDock Vina v1.2.7
#################################################################
# If you used AutoDock Vina in your work, please cite:          #
#                                                               #
# J. Eberhardt, D. Santos-Martins, A. F. Tillack, and S. Forli  #
# AutoDock Vina 1.2.0: New Docking Methods, Expanded Force      #
# Field, and Python Bindings, J. Chem. Inf. Model. (2021)       #
# DOI 10.1021/acs.jcim.1c00203                                  #
#                                                               #
# O. Trott, A. J. Olson,                                        #
# AutoDock Vina: improving the speed and accuracy of docking    #
# with a new scoring function, efficient optimization and       #
# multithreading, J. Comp. Chem. (2010)                         #
# DOI 10.1002/jcc.21334                                         #
#                                                               #
# Please see https:

,pose,vina_total_kcal_mol
0,1,-9.987
1,2,-9.073
2,3,-8.886
3,4,-8.822
4,5,-8.682
5,6,-8.561
6,7,-8.468
7,8,-8.413
8,9,-8.408


## 3. Convert docked poses to SDF with Meeko

Meeko uses SMILES and atom-index information embedded in the ligand PDBQT to restore chemical connectivity and explicit coordinates. If conversion fails, stop and share the printed output rather than using arbitrary PDBQT atom order.

In [5]:
export_sdf=out/'1LT_redocked_all_poses.sdf'
cmd=[sys.executable,'-m','meeko.cli.mk_export',str(poses_path),'-s',str(export_sdf)]
p=subprocess.run(cmd,capture_output=True,text=True)
(out/'meeko_export.log').write_text(' '.join(cmd)+'\n'+p.stdout+'\n'+p.stderr)
print('Export exit status:',p.returncode)
print((p.stdout+'\n'+p.stderr)[-3500:])
if p.returncode or not export_sdf.exists() or export_sdf.stat().st_size==0:
    raise RuntimeError('Meeko SDF export failed; do not claim redocking validation.')
poses=[m for m in Chem.SDMolSupplier(str(export_sdf), removeHs=True) if m is not None]
print('Converted SDF poses:',len(poses))
if not poses:raise RuntimeError('No readable docked SDF poses')

Export exit status: 0


Converted SDF poses: 9


## 4. Atom-mapped heavy-atom RMSD — **no superposition**

Both structures are already in the same crystallographic reference coordinate frame. To assess pose recovery, coordinates **must not be optimally aligned** before RMSD. Atom mapping is determined from molecular graph isomorphism, allowing chemically equivalent atom permutations (e.g., symmetry). Failure to find a full mapping is a stop condition; do not use order-by-index matching as a substitute.

In [6]:
ref=Chem.RemoveHs(reference)
ref_xyz=np.array(ref.GetConformer().GetPositions())
assert ref.GetNumAtoms()==30

def unmoved_symmetry_rmsd(ref,docked,max_matches=10000):
    dock=Chem.RemoveHs(docked)
    if dock.GetNumAtoms()!=ref.GetNumAtoms():
        raise RuntimeError(f'Atom count differs: ref {ref.GetNumAtoms()} dock {dock.GetNumAtoms()}')
    # Query is reference graph; returned match maps reference atom index -> docked atom index.
    matches=dock.GetSubstructMatches(ref,uniquify=False,useChirality=False,maxMatches=max_matches)
    if not matches:
        raise RuntimeError('No chemically identical heavy-atom graph mapping. Review protonation/tautomer and bond orders.')
    if len(matches)==max_matches:
        raise RuntimeError('Too many symmetry matches; confirm all mappings enumerated before a minimum RMSD claim.')
    xyz=np.array(dock.GetConformer().GetPositions())
    values=[float(np.sqrt(np.mean(np.sum((xyz[list(mapping)]-ref_xyz)**2,axis=1)))) for mapping in matches]
    j=int(np.argmin(values))
    return values[j],len(matches),matches[j]

rows=[]
for i,m in enumerate(poses,1):
    rmsd,nmaps,_=unmoved_symmetry_rmsd(ref,m)
    energy=float(energies[i-1][0]) if i<=len(energies) else None
    rows.append({'pose':i,'vina_total_kcal_mol':energy,'heavy_atom_RMSD_A':round(rmsd,4),'symmetry_mappings_checked':nmaps})
results=pd.DataFrame(rows)
results.to_csv(out/'redocking_pose_rmsd.csv',index=False)
display(results)
if results.empty:raise RuntimeError('No poses evaluated')
top1=float(results.iloc[0]['heavy_atom_RMSD_A'])
best=float(results['heavy_atom_RMSD_A'].min())
print('TOP-RANKED pose RMSD:',round(top1,3),'Å')
print('LOWEST RMSD among sampled poses:',round(best,3),'Å')
print('TOP POSE:', 'PASS ≤ 2 Å' if top1<=2 else 'NOT PASSING 2 Å')
print('Note: success of any lower-ranked pose does not imply the top-ranked pose passed.')

,pose,vina_total_kcal_mol,heavy_atom_RMSD_A,symmetry_mappings_checked
0,1,-9.987,0.6039,12
1,2,-9.073,1.9509,12
2,3,-8.886,3.0539,12
3,4,-8.822,1.9853,12
4,5,-8.682,5.6341,12
5,6,-8.561,7.5945,12
6,7,-8.468,3.5131,12
7,8,-8.413,3.5912,12
8,9,-8.408,2.3829,12


TOP-RANKED pose RMSD: 0.604 Å
LOWEST RMSD among sampled poses: 0.604 Å
TOP POSE: PASS ≤ 2 Å
Note: success of any lower-ranked pose does not imply the top-ranked pose passed.


## 5. Save validation summary and download output ZIP

**Interpretation:** Approximately ≤2 Å heavy-atom RMSD of the *top-ranked* pose is a conventional pose-recovery benchmark, not an absolute guarantee of reliable virtual screening. Examine preparation exclusions, protonation, structural completeness, binding-site placement, and ligand chirality regardless of the result. If redocking fails, revise the preparation before docking the ten candidate analogs.

In [7]:
report={
 'target':'PI3Kalpha','pdb_id':'4JPS','reference_ligand':'1LT_alpelisib',
 'vina_version':VINA_VERSION,'meeko_version':version('meeko'),
 'center_A':BOX_CENTER,'size_A':BOX_SIZE,'exhaustiveness':EXHAUSTIVENESS,
 'n_poses_requested':N_POSES,'seed':SEED,'scoring_function':'vina',
 'top_ranked_pose_rmsd_A':top1,'best_pose_rmsd_A':best,
 'top_ranked_pass_2A':bool(top1<=2.0),
 'rmsd_definition':'Graph/symmetry-aware heavy-atom RMSD in fixed crystal coordinate frame; NO alignment',
 'receptor_preparation_caveat':'10 incomplete/excess-bond residues excluded >20 Å outside the provisional box; original 85 pocket heavy atoms retained at 0.0 Å displacement (prior verification)',
 'interpretation':'Redocking validation only, not validation of experimental anticancer potency',
 'status':'provisional redocking assessment; inspect output and receptor assumptions before candidate docking'
}
(out/'pi3ka_step6_redocking_report.json').write_text(json.dumps(report,indent=2)+'\n')
archive=shutil.make_archive('PI3Kalpha_4JPS_Step6_Redocking','zip',out)
print('Saved',archive,'size',Path(archive).stat().st_size)
files.download(archive)
print('Upload this ZIP for inspection before starting ten-analog docking.')

Saved /content/PI3Kalpha_4JPS_Step6_Redocking.zip size 250752


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Upload this ZIP for inspection before starting ten-analog docking.
